# Clang AST generation

Clang was chosen over GCC for AST generation because it provides a stable and user-oriented AST API, making it more suitable for program analysis and machine learning workflows.

## Imports

In [ ]:
import json
from pathlib import Path
from gnn_ai_code_detector.preprocess import CCppPreprocessor

## AST preparation

In [2]:
EXAMPLE = Path("./examples/example.c")
CLANG = r"C:\Program Files\LLVM\bin\clang.exe"

In [ ]:
converter = CCppPreprocessor(CLANG)

To generate the AST in JSON format, Clang's -ast-dump=json functionality is invoked. The resulting tree contains a complete syntactic representation of the translation unit, including user-defined code and compiler-generated information.

In [4]:
ast = converter.build_ast(EXAMPLE)

output = Path("examples/add_ast.json")

with output.open("w", encoding="utf-8") as f:
    json.dump(ast, f, indent=2)

print(f"Number of branches: {len(ast["inner"])}")

Number of branches: 433


Besides the code present in the file itself, Clang's AST also contains the branches that represent the resolved includes, which get removed since they are irrelevant to the code structure and would present a *very* large amount of noise. 

In [ ]:
cut_ast = converter.cut_irrelevant_branches(ast)

output = Path("examples/cut_ast.json")

with output.open("w", encoding="utf-8") as f:
    json.dump(cut_ast, f, indent=2)

for branch in cut_ast["inner"]:
    print(branch["kind"], branch.get("name"))

TypedefDecl MyInt
RecordDecl Point
VarDecl global_counter
FunctionDecl add
FunctionDecl helper
FunctionDecl main


Irrelevant metadata used for source mapping and debugging purposes such as node identifiers, source locations and source ranges gets removed for clarity.

In [11]:
clean_ast = converter.remove_metadata(cut_ast)

output = Path("examples/clean_ast.json")

with output.open("w", encoding="utf-8") as f:
    json.dump(clean_ast, f, indent=2)

Clang additionally inserts compiler-generated nodes that do not correspond to explicit constructs in the source code, which get flattened because they add redundancy without useful information.

In [ ]:
clean_ast = converter.remove_irrelevant_nodes(clean_ast)

output = Path("examples/ast_no_artifacts.json")

with output.open("w", encoding="utf-8") as f:
    json.dump(clean_ast, f, indent=2)

# HumanVsAI_CodeDataset

## Imports

In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
import tempfile
from gnn_ai_code_detector.preprocess import CCppPreprocessor
import subprocess
import json
import re

## Dataset exploration

In [3]:
df = pd.read_csv("../../data/Code_Dataset/HumanVsAi_CodeDataset.csv")
df.head()

,problem_id,Sample_Code,Generated,Language,Source
0,Prob3509,\n/* ITP1_2_D */\n\nimport Java.io.*;\nimport ...,Human,Java,CodeNet
1,Prob292,import Java.util.Scanner;\n\npublic class Main...,Human,Java,CodeNet
2,Prob4407,from datetime import datetime\n\ndef get_time_...,AI,Python,ChatGPT-4
3,Prob4805,"N,M = [int(i) for i in input().split()]\nc = [...",Human,Python,CodeNet
4,Prob5079,import Java.util.*;\n\n\npublic class Main {\n...,Human,Java,CodeNet


In [4]:
c_cpp_df = df[df["Language"].isin(["C", "C++"])].copy()

print(f"Total C/C++ samples: {len(c_cpp_df)}")

print(c_cpp_df["Generated"].value_counts())

print(f"Unique problems: {c_cpp_df["problem_id"].nunique()}")

Total C/C++ samples: 4377
Generated
AI       2189
Human    2188
Name: count, dtype: int64
Unique problems: 2847


In [5]:
solution_counts = c_cpp_df.groupby("problem_id")["Generated"].value_counts().unstack(fill_value=0)

only_ai = ((solution_counts["AI"] > 0) & (solution_counts["Human"] == 0)).sum()
only_human = ((solution_counts["Human"] > 0) & (solution_counts["AI"] == 0)).sum()
both = ((solution_counts["AI"] > 0) & (solution_counts["Human"] > 0)).sum()

print(f"Only AI: {only_ai}")
print(f"Only Human: {only_human}")
print(f"Both: {both}")

Only AI: 1465
Only Human: 1382
Both: 0


The C/C++ subset is almost perfectly balanced with respect to authorship, containing 2189 AI-generated and 2188 human-written solutions. The dataset contains 2847 unique problems, with each problem represented by one or more solutions either completely AI-generated or human-written. Since multiple solutions may correspond to the same programming problem, the dataset will be partitioned using `problem_id` as the grouping variable to prevent data leakage.

To check for potential compilation errors, a quick check with no AST generation is performed. A lot of the source files lack the necessary standard headers, which get appended during preprocessing since imports have no influence on the AST structure and allow to compile more files successfully.

In [ ]:
converter = CCppPreprocessor(CLANG)

clang_args = [ # remove AST generation to make the check faster
    arg for arg in converter.CLANG_ARGS
    if arg not in ("-Xclang", "-ast-dump=json")
]

def check_sample(sample):
    sample_id, row = sample

    language = row["Language"]
    extension = ".c" if language == "C" else ".cpp"

    with tempfile.NamedTemporaryFile(
        mode="w", suffix=extension,
        encoding="utf-8", delete=False
    ) as f:
        f.write(converter.append_headers(row["Sample_Code"], language))
        path = Path(f.name)

    try:
        subprocess.run(
            [converter.clang_path, *clang_args, str(path)],
            capture_output=True, text=True, check=True
        )

        return None

    except subprocess.CalledProcessError as e:
        return {
            "sample_id": sample_id, "problem_id": row["problem_id"],
            "language": language, "generated": row["Generated"],
            "error": e.stderr, "code": row["Sample_Code"]
        }

    finally:
        path.unlink(missing_ok=True)

max_workers = 8

samples = c_cpp_df.iterrows()

from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    results = executor.map(check_sample, samples)

failures = [result for result in results if result is not None]

with open("failures.json", "w", encoding="utf-8") as outfile:
    json.dump(failures, outfile, indent=2, ensure_ascii=False)

total = len(c_cpp_df)
failed = len(failures)

print(
    f"total={total:4} "
    f"failed={failed:3} "
    f"success={(total - failed):4} "
    f"failure_rate={100*failed / total:.2f}%"
)

total=4377 failed=531 success=3846 failure_rate=12.13%


A heuristic is used to determine error types.

In [11]:
def categorize_failure(error: str) -> str:
    error = error.lower()

    error_patterns = {
        "file not found": "missing_header",
        "unknown type name 'import'": "wrong_language",
        "expected ';'": "syntax_error",
        "expected expression": "syntax_error",
        "expected unqualified-id": "syntax_error",
        "redefinition": "semantic_error",
        "incompatible": "type_error",
        "undeclared": "missing_declaration",
        "not marked const": "const_correctness",
    }

    for pattern, category in error_patterns.items():
        if pattern in error:
            return category

    if "expected" in error and "error:" in error:
        return "syntax_error"

    return "other"
categories = Counter(
    categorize_failure(f["error"])
    for f in failures
)

for category, count in categories.most_common():
    print(f"{count:4} | {category}")

 193 | syntax_error
 107 | missing_header
 106 | other
  92 | missing_declaration
  13 | semantic_error
  10 | const_correctness
   6 | type_error
   4 | wrong_language


In [12]:
grouped = defaultdict(list)

for failure in failures:
    category = categorize_failure(failure["error"])
    grouped[category].append(failure)

for category, samples in grouped.items():
    counts = Counter(s["generated"] for s in samples)

    print(
        f"{category:20} "
        f"total={len(samples):3} "
        f"AI={counts['AI']:3} "
        f"Human={counts['Human']:2}"
    )

syntax_error         total=193 AI=127 Human=66
missing_header       total=107 AI= 89 Human=18
missing_declaration  total= 92 AI= 73 Human=19
other                total=106 AI= 57 Human=49
type_error           total=  6 AI=  4 Human= 2
wrong_language       total=  4 AI=  1 Human= 3
const_correctness    total= 10 AI= 10 Human= 0
semantic_error       total= 13 AI=  6 Human= 7


Interestingly, AI appears to make significantly more syntax errors and misses more headers (or maybe uses more obscure headers) than humans. These diagnostics are not used as features by the classifier and are retained solely for dataset quality analysis and troubleshooting. Samples with no usable AST are excluded from graph construction.

In [13]:
missing_headers = Counter()

for failure in grouped["missing_header"]:
    matches = re.findall(
        r"fatal error: ['\"]([^'\"]+)['\"] file not found",
        failure["error"]
    )
    
    for header in matches:
        missing_headers[header] += 1

for header, count in missing_headers.most_common()[:10]:
    print(f"{count:4} | {header}")

  14 | cstdio
  12 | cstdint
  10 | arpa/inet.h
   8 | QApplication
   7 | matplotlibcpp.h
   6 | bitcoin/bitcoin.hpp
   6 | climits
   3 | jwt-cpp/jwt.h
   3 | xlnt/xlnt.hpp
   3 | QMessageBox


Upon inspecting the missing headers closer, it becomes apparent that the majority of it comes from mislabeling (C++ code marked as C that fails to compile due to to including `cstdio` and `cstdint`). The failures that are due to the compiler setup (specific or obscure libraries missing) are too few and far between to make fixing them worth the time.

In [14]:
total = Counter(c_cpp_df["Generated"])
failed = Counter(f["generated"] for f in failures)

for label in ["AI", "Human"]:
    n_total = total[label]
    n_failed = failed[label]
    n_success = n_total - n_failed

    print(
        f"{label:6} "
        f"total={n_total:4} "
        f"failed={n_failed:3} "
        f"success={n_success:4} "
        f"failure_rate={100*n_failed/n_total:.2f}%"
    )

AI     total=2189 failed=367 success=1822 failure_rate=16.77%
Human  total=2188 failed=164 success=2024 failure_rate=7.50%


Out of all C/C++ samples, 12.13% failed to produce a usable AST, leaving 3,871 samples (88.44%) for graph construction. The failure rate is higher for AI-generated (16.77%) than for human-written samples (7.50%). Since compilation diagnostics are not provided to the classifier, failed samples are excluded rather than used as model features. This difference is retained as a dataset-quality consideration and does not affect the graph representation. After accounting for compilation failures, the AI/human generated code class split becomes 47.4%/52.6%, which is reasonably balanced and doesn't call for any additional measures.